# Partial Differential Equations

&nbsp;[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/themintlab/ExecutableEngineering/blob/main/chapters/differential_equations/partial_differential_equations/partial_differential_equations.ipynb)

Partial differential equations (PDEs) expand ordinary differential equations (ODEs) to include multiple independent variables (coordinates):

* Additional *spatial* coordinates are typically extensions of the ODE case and are usually posed and treated as boundary value problems (BVPs).
* The *time* coordinate is almost always posed as an initial value problem (IVP) and tends to introduce complexity in the form of numerical stability constraints.

### Classification of PDEs

Introducing time as a coordinate in PDEs raises the fundamental question of *how information propagates* from the initial condition or from events that occur during the solution.

In [ ]:
try:
    import executable_engineering as exe
except ImportError:
    %pip install -q executable_engineering
    import executable_engineering as exe

## Elliptic Equations

Second-order equations of the form (with $B^2 - 4AC < 0$):

$$
A\frac{\partial^2 u}{\partial x^2} + B\frac{\partial^2 u}{\partial x \partial y} + C\frac{\partial^2 u}{\partial y^2} = f(x, y)
$$

are termed *elliptic*. The absence of a time-derivative term implies that information is *not being propagated* over time, but rather is instantaneously available everywhere in space—all transients have decayed and the system has reached a steady state (even if coefficients such as $A(t)$ vary quasi-statically).

* **Laplace's and Poisson's equations** ($\nabla^2 u = 0$, $\nabla^2 u = f(\mathbf{x})$): electrostatic potential with charge density, steady-state heat conduction with a source, or velocity potential of an incompressible, irrotational fluid.
* **Helmholtz equation** ($\nabla^2 u + k^2 u = 0$): acoustic vibrations of a membrane or cavity, and quantum-mechanical wave functions in a potential well (obtained after separating time and space variables).

### Solution Approaches

Solving elliptic equations requires solving a coupled system of algebraic (possibly nonlinear) equations simultaneously across the entire spatial mesh:

* **Initial guess**: Nonlinear root finders require a sufficiently close initial guess. A common mitigation strategy is *continuation* (parametrically ramping up a physical parameter, such as gradually increasing the heat load in natural convection).
* **Convergence**: Discretizing a fine mesh yields a high-dimensional system where root finders may wander into local minima or leave the feasible region; trust-region methods help mitigate this.
* **Parallelization**: Under *domain decomposition*, elliptic coupling implies every subdomain instantaneously influences all others, creating heavy communication overhead unless accelerated by *multigrid methods* and *preconditioners*.

## Parabolic Equations

Equations of the form:

$$
A \frac{\partial u}{\partial t} - C \frac{\partial^2 u}{\partial x^2} = 0
$$

propagate information in one temporal direction (forward in time) and govern *diffusive* processes. Diffusion naturally smooths out sharp discontinuities and high-frequency spatial oscillations, though numerical time-stepping can still suffer from severe stability restrictions.

### Solution Schemes

Diffusion problems are solved via a *time-marching scheme* in which time is discretized as an IVP while the spatial operator is discretized as a BVP at each step:

* **Stiffness**: Fine spatial grids introduce widely separated decay time scales, making parabolic PDEs stiff.
* **Ill-posed inverse problems**: When back-calculating model parameters or initial states from later measurements (e.g., inferring thermal conductivity from a cooling profile), the smoothing nature of forward diffusion works against us—many distinct initial states diffuse into nearly identical smooth profiles.

## Hyperbolic Equations

Equations of the form:

$$
A \frac{\partial^2 u}{\partial t^2} - C \frac{\partial^2 u}{\partial x^2} = 0
$$

propagate information both forward *and* backward along characteristic lines at a finite wave speed. Replacing $t$ with $-t$ leaves the governing equation invariant (*time-reversal symmetry*). Hyperbolic PDEs describe wave propagation (second-order) as well as pure advection (first-order, e.g., $A \frac{\partial u}{\partial t} + C \frac{\partial u}{\partial x} = 0$).

### Solution Schemes

* **Reduction of order**: Second-order time derivatives can be decomposed into a coupled system of first-order equations in time.
* **Spectral methods**: Systems with periodic boundary conditions are well-suited to Fourier spectral analysis, which avoids artificial numerical dispersion.
* **Reversibility**: Time-reversal symmetry makes hyperbolic systems amenable to inverse reconstruction of earlier states from later wave measurements.

## Solving Time-Dependent PDEs

The standard approach to solving time-dependent PDEs combines IVP time-marching with spatial BVP discretization:

1. Discretize the time derivative as an IVP relating the new state $T^{t+1}$ to the known state $T^t$.
2. Discretize the spatial derivatives on a mesh of spacing $\Delta x$ using finite differences.
3. March forward in time, enforcing boundary conditions at each time step.

Consider the 1D parabolic heat equation with thermal diffusivity $\alpha > 0$:

$$
\frac{\partial T}{\partial t} = \alpha \frac{\partial^2 T}{\partial x^2}, \qquad \alpha \frac{\partial^2 T}{\partial x^2} \approx \alpha \frac{T_{i-1} - 2 T_i + T_{i+1}}{\Delta x^2}
$$

### Notation

* Spatial grid points are indexed with a subscript $i$ ($x_i = i \Delta x$).
* Discrete time levels are indexed with a superscript $t$, marching from known $T_i^t$ to unknown $T_i^{t+1}$.

## Explicit Euler Time-Stepping

Evaluating the spatial second derivative at the *current* time level $t$ yields the explicit (Forward Euler) scheme:

$$
\begin{aligned}
\frac{T^{t+1}_i - T^{t}_i}{\Delta t} &= \alpha \frac{T^t_{i-1} - 2 T^t_i + T^t_{i+1}}{\Delta x^2} \\
T^{t+1}_i &= T_i^t + r \left(T^t_{i-1} - 2 T^t_i + T^t_{i+1}\right), \qquad r = \frac{\alpha \Delta t}{\Delta x^2}
\end{aligned}
$$

forward Euler stencil.svg

Each grid point $T_i^{t+1}$ at the new time step depends *only* on three known values at time level $t$. Use the slider below to increase $\Delta t$ past the stability threshold $r = 0.5$.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def forward_euler_heat
:end-before: def backward_euler_heat
```

In [ ]:
exe.display_source(exe.forward_euler_heat)
fig = exe.plot_forward_euler_heat()
fig.show()

## Von Neumann Stability Analysis

To determine when spatial oscillations grow or decay, *von Neumann stability analysis* expands the spatial profile in Fourier modes ${}^k T_i^t = e^{j k i \Delta x}$ (with wave number $k$ and $j = \sqrt{-1}$) and defines the amplification factor $G$ via ${}^k T_i^{t+1} = G \, {}^k T_i^t$. Stability requires $|G| \le 1$ for all wave numbers $k$.

Substituting a Fourier mode into the Forward Euler update gives:

$$
\begin{aligned}
{}^k T^{t+1}_i &= e^{j k i \Delta x} + r \left(e^{j k (i-1) \Delta x} - 2 e^{j k i \Delta x} + e^{j k (i+1) \Delta x}\right) \\
&= {}^k T^{t}_i \left[1 + r \left(e^{j k \Delta x} + e^{-j k \Delta x} - 2\right)\right] \\
\implies G &= 1 + 2 r \left[\cos(k \Delta x) - 1\right]
\end{aligned}
$$

Since $\cos(k \Delta x) \in [-1, 1]$, the amplification factor lies in $[1 - 4r, 1]$. Requiring $1 - 4r \ge -1$ yields the 1D stability limit (and its 2D/3D extensions on a uniform grid of spacing $h = \Delta x$):

$$
\text{1D: } \Delta t \le \frac{\Delta x^2}{2 \alpha}, \qquad \text{2D: } \Delta t \le \frac{h^2}{4 \alpha}, \qquad \text{3D: } \Delta t \le \frac{h^2}{6 \alpha}
$$

Because $\Delta t$ scales with $\Delta x^2$, halving the grid spacing forces a fourfold reduction in time step—a severe stiffness penalty shared by explicit diffusion schemes.

## Implicit Euler Time-Stepping

Evaluating the spatial finite difference at the *new* time level $t+1$ yields the Backward (Implicit) Euler scheme:

$$
-r T^{t+1}_{i-1} + (1 + 2r) T^{t+1}_i - r T^{t+1}_{i+1} = T_i^{t}
$$

backward Euler stencil.svg

Von Neumann analysis gives the amplification factor $G = \frac{1}{1 + 2r[1 - \cos(k \Delta x)]} \in \left[\frac{1}{1 + 4r}, 1\right]$, which satisfies $|G| \le 1$ for *all* $\Delta t > 0$. Backward Euler is **unconditionally stable**, at the cost of solving a tridiagonal linear system $\mathbf{A} \mathbf{T}^{t+1} = \mathbf{T}^t$ at each time step.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def backward_euler_heat
:end-before: def crank_nicolson_heat
```

In [ ]:
exe.display_source(exe.backward_euler_heat)
fig = exe.plot_backward_euler_heat()
fig.show()

## Crank-Nicolson Method

Backward Euler is only first-order accurate in time ($\mathcal{O}(\Delta t)$), whereas central differencing in space is second-order ($\mathcal{O}(\Delta x^2)$). The **Crank-Nicolson method** achieves second-order accuracy in both time and space by averaging the explicit and implicit spatial discretizations at levels $t$ and $t+1$:

$$
\begin{aligned}
\frac{T^{t+1}_i - T^{t}_i}{\Delta t} &= \frac{\alpha}{2 \Delta x^2} \left[\left(T^{t+1}_{i-1} - 2 T^{t+1}_i + T^{t+1}_{i+1}\right) + \left(T^{t}_{i-1} - 2 T^{t}_i + T^{t}_{i+1}\right)\right] \\
-r T_{i-1}^{t+1} + (1 + 2r) T_i^{t+1} - r T_{i+1}^{t+1} &= r T_{i-1}^{t} + (1 - 2r) T_i^{t} + r T_{i+1}^{t}, \qquad r = \frac{\alpha \Delta t}{2 \Delta x^2}
\end{aligned}
$$

Crank Nicholson stencil.svg

The left-hand matrix $\mathbf{A}$ has the *exact same tridiagonal sparsity* as Backward Euler, delivering $\mathcal{O}(\Delta t^2 + \Delta x^2)$ accuracy and unconditional stability for virtually identical computational cost per step.

```{literalinclude} ../../../executable_engineering/executable_engineering/differential_equations_module.py
:language: python
:start-at: def crank_nicolson_heat
:end-before: crank_nicholson_heat =
```

In [ ]:
exe.display_source(exe.crank_nicolson_heat)
fig = exe.plot_pde_method_comparison()
fig.show()

## Summary

Implicit time-steppers are invaluable for stiff parabolic PDEs, but each time step requires solving a coupled system of equations. For large 2D/3D meshes, direct factorization ($\mathbf{L}\mathbf{U}$) becomes prohibitively expensive, necessitating iterative linear solvers.

Explicit time-steppers are conditionally stable—which restricts $\Delta t$—but require no simultaneous equation solving or nonlinear root finding:

* No dependence on initial guesses for root finders
* Guaranteed per-step update without convergence failures
* Trivial parallelization across high-performance computing (HPC) nodes
* Minimal memory footprint (no system matrix assembly)

When does the cost of implicit linear/nonlinear iterations outweigh the small time-step restriction of an explicit method? *It depends on the problem:*

* **General multiphysics solvers** (e.g., COMSOL, ANSYS) typically default to implicit time-stepping to handle coupled stiff physics over long time scales.
* **Specialized short-duration or massive-mesh solvers** often prefer explicit methods that advance millions of degrees of freedom without matrix assembly—such as **LS-DYNA** (high-velocity impact, crash, and drop tests) and explicit **CFD packages** (e.g., FLOW-3D).